# XGBoost Experiments

**Objective:**

This notebook evaluates the performance of the XGBoost model using different strategies to handle class imbalance:

- Baseline
- Class-weight
- SMOTE
- SMOTEENN

**All experiments use:**
- A shared preprocessing pipeline
- Stratified K-Fold cross-validation
- A unified evaluation framework

Results are stored in a shared results table for comparison with other models.

## Setup

In [ ]:
# Add project root to system path so we can import from src/
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent.parent / "bank-deposit-prediction"
sys.path.append(str(PROJECT_ROOT))


# Standard libraries
import pandas as pd

# Modeling
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

# Imbalanced learning tools
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTEENN

# Shared project modules
from src.shared import get_cv, get_preprocessing_steps
from src.evaluation import evaluate_model, save_results



## Data Loading

In [ ]:
data_path = PROJECT_ROOT / "data" / "raw" / "bank2.csv"

df = pd.read_csv(data_path, sep=';')

y = df['y'].map({'yes': 1, 'no': 0})
X = df.drop(columns=['y'])

## Shared components

We use:
- A centralized preprocessing pipeline (no data leakage)
- Stratified K-Fold cross-validation for fair evaluation

In [3]:
# Get shared cross-validation strategy
cv = get_cv()

## Baseline XGBoost

- No imbalance handling
- Serves as a reference model

In [ ]:
all_results = []

# Build baseline pipeline
baseline_pipeline = Pipeline(
    get_preprocessing_steps() +  # Apply preprocessing
    [('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ))
])

# Evaluate model
result_baseline  = evaluate_model("XGBoost", "Baseline", baseline_pipeline, X, y, cv)
all_results.append(result_baseline)
result_baseline

{'Model': 'XGBoost',
 'Strategy': 'Baseline',
 'Accuracy': np.float64(0.883),
 'Precision': np.float64(0.4765),
 'Recall': np.float64(0.1784),
 'F1': np.float64(0.259),
 'ROC-AUC': np.float64(0.702),
 'PR-AUC': np.float64(0.3037)}

## Class Weight (Imbalance Handling)

- Adjusts model to penalize misclassification of minority class
- Uses scale_pos_weight

In [5]:
# Compute class weight ratio
scale_pos_weight = (y == 0).sum() / (y == 1).sum()

cw_pipeline = Pipeline(
    get_preprocessing_steps() +
    [('model', XGBClassifier(
        eval_metric='logloss',
        scale_pos_weight=scale_pos_weight,
        random_state=42
    ))
])

result_cw = evaluate_model("XGBoost", "ClassWeight", cw_pipeline, X, y, cv)
all_results.append(result_cw)
result_cw

{'Model': 'XGBoost',
 'Strategy': 'ClassWeight',
 'Accuracy': np.float64(0.8498),
 'Precision': np.float64(0.33),
 'Recall': np.float64(0.2936),
 'F1': np.float64(0.3099),
 'ROC-AUC': np.float64(0.6806),
 'PR-AUC': np.float64(0.2791)}

## SMOTE (Oversampling)

- Generates synthetic samples for the minority class
- Applied inside the pipeline to avoid data leakage

In [6]:
smote_pipeline = ImbPipeline(
    get_preprocessing_steps() +
    [('smote', SMOTE(random_state=42)),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ))
])

result_smote = evaluate_model("XGBoost", "SMOTE", smote_pipeline, X, y, cv)
all_results.append(result_smote)
result_smote

{'Model': 'XGBoost',
 'Strategy': 'SMOTE',
 'Accuracy': np.float64(0.8834),
 'Precision': np.float64(0.4829),
 'Recall': np.float64(0.1765),
 'F1': np.float64(0.2567),
 'ROC-AUC': np.float64(0.7172),
 'PR-AUC': np.float64(0.3178)}

## SMOTEENN (Hybrid Method)

- Combines:
  - SMOTE (oversampling)
  - ENN (cleaning noisy samples)
- Aims to improve class balance and data quality

In [7]:
smoteenn_pipeline = ImbPipeline(
    get_preprocessing_steps() +
    [('smoteenn', SMOTEENN(random_state=42)),
    ('model', XGBClassifier(
        eval_metric='logloss',
        random_state=42
    ))
])

result_smoteenn = evaluate_model("XGBoost", "SMOTEENN", smoteenn_pipeline, X, y, cv)
all_results.append(result_smoteenn)

result_smoteenn

{'Model': 'XGBoost',
 'Strategy': 'SMOTEENN',
 'Accuracy': np.float64(0.829),
 'Precision': np.float64(0.3266),
 'Recall': np.float64(0.4549),
 'F1': np.float64(0.38),
 'ROC-AUC': np.float64(0.727),
 'PR-AUC': np.float64(0.3323)}

## Results Summary

All experiment results are stored in a shared table for comparison with other models

In [8]:
# Save results
results_df = pd.DataFrame(all_results)
save_results(results_df)

# Read experiment_results table
results_path = PROJECT_ROOT / "results" / "experiment_results.csv"
results = pd.read_csv(results_path)
results

,Model,Strategy,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC
0,XGBoost,Baseline,0.8830,0.4765,0.1784,0.2590,0.7020,0.3037
1,XGBoost,ClassWeight,0.8498,0.3300,0.2936,0.3099,0.6806,0.2791
2,XGBoost,SMOTE,0.8834,0.4829,0.1765,0.2567,0.7172,0.3178
3,XGBoost,SMOTEENN,0.8290,0.3266,0.4549,0.3800,0.7270,0.3323
